# Task 15 - Enterprise LLM Gateway with Dynamic Rate-Limiting and Fallback Governance
## Stack: FastAPI, Redis, Docker, Prometheus, Grafana

This notebook demonstrates an Enterprise LLM Gateway supporting dynamic rate-limiting (token bucket algorithm), model fallback governance (routing and failover), and observability (Prometheus & Grafana) using FastAPI.

### Architecture Layout (ASCII)
```
 +-------------+       +-------------------+       +-----------------------+
 |             | ----> |                   | ----> |                       |
 |   Clients   |       |  FastAPI Gateway  |       |   Primary LLM Model   |
 |             | <---- |                   | <---- | (OpenAI, Azure, etc.) |
 +-------------+       +-------------------+       +-----------------------+
                              |    ^                            ^
  Rate Limits & Metrics       |    | Route / Fallback           |
  +----------------------+    |    |                            |
  |                      | <--+    v                            |
  | - Redis (Tokens)     |   +-----------------------+          | Fallback / Failover
  | - Prometheus (Stats) |   |                       |          |
  |                      |   |   Fallback LLM Model  | ---------+
  +----------------------+   +-----------------------+
```

In [ ]:
import time
import asyncio
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from datetime import datetime

# Set seed for reproducible mock testing
random.seed(42)
np.random.seed(42)

print("Required modules loaded successfully.")


In [ ]:
class TokenBucketRateLimiter:
    """
    In-memory TokenBucketRateLimiter mimicking Redis token logic.
    Manages rate limits dynamically.
    """
    def __init__(self, capacity: int, refill_rate: float):
        # The maximum number of tokens the bucket can hold (burst capacity)
        self.capacity = capacity
        # The rate at which tokens are added per second (sustained rate)
        self.refill_rate = refill_rate
        # The current number of tokens available in the bucket
        self.tokens = capacity
        # The timestamp of the last token refill, initialized to current time
        self.last_refill = time.time()
        
    def _refill(self):
        # Get the current time for refill calculation
        now = time.time()
        # Calculate the time elapsed since the last refill operation
        elapsed = now - self.last_refill
        # Determine how many tokens to add based on elapsed time and refill rate
        tokens_to_add = elapsed * self.refill_rate
        
        # If there's at least a fraction of a token to add
        if tokens_to_add > 0:
            # Update the bucket's tokens, not exceeding maximum capacity
            self.tokens = min(self.capacity, self.tokens + tokens_to_add)
            # Update the last refill timestamp to now
            self.last_refill = now
            
    def consume(self, tokens: int = 1) -> tuple[bool, float]:
        """
        Consumes tokens from the bucket.
        Returns (success_boolean, retry_after_seconds)
        """
        # Trigger a refill before attempting to consume tokens
        self._refill()
        
        # Check if we have enough tokens to fulfill the request (burst checking)
        if self.tokens >= tokens:
            # Subtract the consumed tokens from the bucket
            self.tokens -= tokens
            # Request granted, retry-after is 0 since it succeeded
            return True, 0.0
        else:
            # Calculate deficit (tokens needed - tokens available)
            deficit = tokens - self.tokens
            # Calculate how long until the deficit is met by the refill rate
            retry_after = deficit / self.refill_rate
            # Request denied, provide the retry-after delay
            return False, retry_after

# Example Usage:
limiter = TokenBucketRateLimiter(capacity=10, refill_rate=2.0)
print("Token Bucket Initialized with capacity 10 and refill rate 2 tokens/sec.")


In [ ]:
class ModelRouter:
    """
    Manages LLM backend statuses, routing, detects error responses (429/503),
    and handles recovery cooldowns and fallback routing.
    """
    def __init__(self, primary_model: str, fallback_models: list):
        # Store the primary model name
        self.primary_model = primary_model
        # Store the list of fallback models to try if primary fails
        self.fallback_models = fallback_models
        # Dictionary tracking the status of each model (True = healthy, False = unhealthy)
        self.statuses = {m: True for m in [primary_model] + fallback_models}
        # Dictionary tracking the timestamp when a model was marked as unhealthy
        self.cooldown_starts = {m: 0.0 for m in [primary_model] + fallback_models}
        # The duration (in seconds) a model remains in cooldown before retry
        self.cooldown_period = 2.0  # short for testing
        
    def _check_cooldowns(self):
        # Iterate over all registered models
        for model in self.statuses:
            # If the model is currently marked as unhealthy (False)
            if not self.statuses[model]:
                # Calculate time spent in cooldown
                elapsed = time.time() - self.cooldown_starts[model]
                # If the elapsed time exceeds the cooldown period
                if elapsed >= self.cooldown_period:
                    # Model has recovered, mark it as healthy again
                    self.statuses[model] = True
                    
    def get_best_model(self) -> str:
        """
        Returns the name of the best available model, routing to fallbacks if needed.
        """
        # First, process any models that might have recovered from cooldown
        self._check_cooldowns()
        
        # Check if the primary model is healthy
        if self.statuses[self.primary_model]:
            # Primary is good, route to it
            return self.primary_model
            
        # Primary is down, iterate through fallbacks in order
        for f_model in self.fallback_models:
            # Check if this fallback model is healthy
            if self.statuses[f_model]:
                # Fallback is good, route to it
                return f_model
                
        # If all models (primary + fallbacks) are down, return None
        return None
        
    def report_error(self, model: str, status_code: int):
        """
        Detects error responses (e.g., 429/503) and marks model for recovery cooldown.
        """
        # If the error code is related to rate-limits (429) or server unreachability (503)
        if status_code in [429, 503]:
            # Mark the model as unhealthy
            self.statuses[model] = False
            # Record the exact time the model went down for cooldown tracking
            self.cooldown_starts[model] = time.time()

# Example usage:
router = ModelRouter(primary_model="GPT-4", fallback_models=["GPT-3.5", "Claude-3-Haiku"])
print("Model Router initialized with primary GPT-4 and fallbacks GPT-3.5, Claude-3-Haiku.")


In [ ]:
class PrometheusMetrics:
    """
    Mock Prometheus Instrumentation (Counters, Gauges, Histograms).
    """
    def __init__(self):
        # Counter to track total requests processed
        self.requests_total = 0
        # Counter to track rate limit rejections
        self.rate_limited_total = 0
        # Counter to track successful responses
        self.success_total = 0
        # Counter to track fallback usages
        self.fallback_total = 0
        # Histogram to store request latencies in seconds
        self.latency_histogram = []
        
    def inc_requests(self):
        # Increment total requests counter
        self.requests_total += 1
        
    def inc_rate_limited(self):
        # Increment rate limit rejection counter
        self.rate_limited_total += 1
        
    def inc_success(self):
        # Increment total success counter
        self.success_total += 1
        
    def inc_fallback(self):
        # Increment fallback routing counter
        self.fallback_total += 1
        
    def observe_latency(self, duration: float):
        # Append the latency observation to the histogram
        self.latency_histogram.append(duration)

metrics = PrometheusMetrics()
print("Prometheus Metrics initialized.")


In [ ]:
class MockFastAPIGateway:
    """
    Mock FastAPI microservice implementation utilizing Rate Limiter and Router modules.
    """
    def __init__(self, rate_limiter, router, metrics):
        # Inject the rate limiter dependency
        self.rate_limiter = rate_limiter
        # Inject the model router dependency
        self.router = router
        # Inject the Prometheus metrics dependency
        self.metrics = metrics
        
    async def handle_request(self, client_id: int):
        # Start timing the request for latency metrics
        start_time = time.time()
        # Increment total requests counter
        self.metrics.inc_requests()
        
        # Step 1: Rate Limiting (FastAPI Middleware logic)
        # Attempt to consume 1 token from the bucket
        allowed, retry_after = self.rate_limiter.consume(1)
        if not allowed:
            # Request denied, increment rate limit counter
            self.metrics.inc_rate_limited()
            # Observe the latency (very short for rejected requests)
            self.metrics.observe_latency(time.time() - start_time)
            # Return early with 429 Too Many Requests
            return {"status": 429, "error": "Rate Limited", "retry_after": retry_after, "model": None}
            
        # Step 2: Routing
        # Get the best available model to handle this request
        target_model = self.router.get_best_model()
        
        if target_model is None:
            # No models are healthy, return 503 Service Unavailable
            self.metrics.observe_latency(time.time() - start_time)
            return {"status": 503, "error": "All models are down", "model": None}
            
        if target_model != self.router.primary_model:
            # If routing to a fallback model, increment the fallback counter
            self.metrics.inc_fallback()
            
        # Step 3: Mock LLM Call
        # Simulate network delay for the LLM response (50ms - 250ms)
        await asyncio.sleep(random.uniform(0.05, 0.25))
        
        # Simulate potential model failure (10% chance)
        is_success = random.random() > 0.1
        
        if is_success:
            # Call succeeded, increment success counter
            self.metrics.inc_success()
            # Observe latency for successful call
            self.metrics.observe_latency(time.time() - start_time)
            # Return 200 OK
            return {"status": 200, "model": target_model, "response": "LLM Output"}
        else:
            # Call failed due to simulated model instability (e.g., 503)
            # Report the error to the router to trigger cooldown for this model
            self.router.report_error(target_model, 503)
            # Observe latency for failed call
            self.metrics.observe_latency(time.time() - start_time)
            # Return 503 Service Unavailable
            return {"status": 503, "error": "Model failure", "model": target_model}

gateway = MockFastAPIGateway(limiter, router, metrics)
print("Mock FastAPI Gateway initialized.")


In [ ]:
async def run_load_test(num_clients: int, requests_per_client: int, burst: bool):
    """
    Mock concurrent load testing suite simulating normal vs burst workloads.
    """
    # List to store the results of all requests
    results = []
    
    async def client_task(client_id: int):
        for _ in range(requests_per_client):
            # Call the gateway to handle the request
            resp = await gateway.handle_request(client_id)
            # Append the response to the results list
            results.append(resp)
            if burst:
                # For burst workloads, extremely short sleep between requests
                await asyncio.sleep(random.uniform(0.01, 0.05))
            else:
                # For normal workloads, moderate sleep to space out requests
                await asyncio.sleep(random.uniform(0.5, 1.0))
                
    # Create an asyncio task for each simulated client
    tasks = [client_task(i) for i in range(num_clients)]
    
    # Run all client tasks concurrently
    await asyncio.gather(*tasks)
    
    return results

# Reset metrics and components for test
limiter = TokenBucketRateLimiter(capacity=20, refill_rate=15.0)
router = ModelRouter(primary_model="GPT-4", fallback_models=["GPT-3.5"])
metrics = PrometheusMetrics()
gateway = MockFastAPIGateway(limiter, router, metrics)

# Run normal workload: 50 clients, 2 requests each
print("Running normal workload...")
normal_results = await run_load_test(num_clients=50, requests_per_client=2, burst=False)
print(f"Normal workload finished. Total Requests: {len(normal_results)}")

# Run burst workload: 50 clients, 5 requests each
print("\nRunning burst workload...")
burst_results = await run_load_test(num_clients=50, requests_per_client=5, burst=True)
print(f"Burst workload finished. Total Requests: {len(burst_results)}")


In [ ]:
def analyze_and_plot(results, title_prefix):
    """
    Graphs analyzing success rates, latencies (p50, p95, p99), rate-limit events, and routing.
    """
    # Dictionary to aggregate statuses
    status_counts = defaultdict(int)
    # Dictionary to aggregate model usage
    model_usage = defaultdict(int)
    
    # Process each result
    for r in results:
        # Count the HTTP status code (200, 429, 503)
        status_counts[r['status']] += 1
        # If a model was used, count its usage
        if r['model']:
            model_usage[r['model']] += 1
            
    # Create a figure with 1 row and 3 columns for side-by-side graphs
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Status Codes Pie Chart
    labels = [str(k) for k in status_counts.keys()]
    sizes = list(status_counts.values())
    axs[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['#4CAF50', '#F44336', '#FF9800'])
    axs[0].set_title(f'{title_prefix} - Status Codes\n(200=OK, 429=RateLimit, 503=Error)')
    
    # 2. Model Routing Bar Chart
    models = list(model_usage.keys())
    counts = list(model_usage.values())
    axs[1].bar(models, counts, color=['#2196F3', '#9C27B0'])
    axs[1].set_title(f'{title_prefix} - Model Routing')
    axs[1].set_ylabel('Request Count')
    
    # 3. Latency Percentiles
    # Extract latencies from metrics. Note: in a real app, latencies are bound to the specific run.
    # Here, we use the global metrics object to calculate percentiles roughly.
    lats = np.array(metrics.latency_histogram) * 1000 # Convert to ms
    if len(lats) > 0:
        p50 = np.percentile(lats, 50)
        p95 = np.percentile(lats, 95)
        p99 = np.percentile(lats, 99)
        axs[2].bar(['p50', 'p95', 'p99'], [p50, p95, p99], color=['#FFC107', '#FF9800', '#FF5722'])
        axs[2].set_title(f'{title_prefix} - Latency (ms)')
        axs[2].set_ylabel('Latency (ms)')
        
    plt.tight_layout()
    plt.show()

# Display analysis for both workloads
print("--- Normal Workload Analysis ---")
analyze_and_plot(normal_results, "Normal")
print("--- Burst Workload Analysis ---")
analyze_and_plot(burst_results, "Burst")


### Production Configurations

**1. `docker-compose.yml`**
```yaml
version: '3.8'
services:
  gateway:
    build: .
    ports:
      - "8000:8000"
    environment:
      - REDIS_URL=redis://redis:6379/0
    depends_on:
      - redis

  redis:
    image: redis:alpine
    ports:
      - "6379:6379"

  prometheus:
    image: prom/prometheus:latest
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    ports:
      - "9090:9090"

  grafana:
    image: grafana/grafana:latest
    ports:
      - "3000:3000"
    depends_on:
      - prometheus
```

**2. `prometheus.yml`**
```yaml
global:
  scrape_interval: 10s

scrape_configs:
  - job_name: 'fastapi-gateway'
    metrics_path: '/metrics'
    static_configs:
      - targets: ['gateway:8000']
```

**3. Grafana Dashboard Config (Conceptual JSON snippet)**
```json
{
  "title": "LLM Gateway Monitoring",
  "panels": [
    {
      "title": "Request Rate (RPS)",
      "type": "graph",
      "targets": [
        { "expr": "rate(requests_total[1m])" }
      ]
    },
    {
      "title": "429 Rate Limit Errors",
      "type": "stat",
      "targets": [
        { "expr": "rate(rate_limited_total[1m])" }
      ]
    },
    {
      "title": "P99 Latency (ms)",
      "type": "gauge",
      "targets": [
        { "expr": "histogram_quantile(0.99, rate(latency_histogram_bucket[5m]))" }
      ]
    }
  ]
}
```